# miniLLM 49M — d'où vient le gain de connaissances ? (comparaison appariée)

Au test de connaissances de 204 questions, les trois entraînements de la recette C (graine 1337, 1, 2) sortent à **18-21 %**, contre **10,8 %** pour le modèle publié. Mais ils diffèrent de lui par **deux choses à la fois** : un pré-entraînement plus long (val_loss 3,20 → 3,04) **et** une recette de SFT différente.

Ce notebook sépare les deux :
- **A** = ancienne recette de SFT (celle du v3) sur le **nouveau** pré-entraînement. Si A ≈ C, le gain vient du pré-entraînement. Si A ≈ PUB, il vient de la recette.
- **B**, **D** : autres variantes de l'ablation, sur le nouveau pré-entraînement.

La comparaison est **appariée** (test exact de McNemar sur les mêmes questions) : plus fine que de comparer des intervalles de confiance.

Durée : ~12 min sur T4, aucun entraînement. **Exécution → GPU (T4) → Tout exécuter.** Les modèles déjà évalués sont relus (cache), une interruption ne fait rien perdre.

In [ ]:
# ── 1) Drive, code, modèle publié ──
import os, sys, glob, shutil, json
from google.colab import drive
drive.mount("/content/drive")

cands = (["/content/drive/MyDrive/MiniLLM_v2"] + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2")
         + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data")), None)
if BASE is None:
    raise SystemExit("Dossier MiniLLM_v2 introuvable (Drive > Partagés > appui long > Ajouter un raccourci > Mon Drive).")
DATA, CKPT = f"{BASE}/data", f"{BASE}/checkpoints"
CODE_DIR = "/content/minillm_v2"

!pip -q install tokenizers datasets huggingface_hub
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} fetch -q origin v2.1.1 && git -C {CODE_DIR} checkout -q v2.1.1 && git -C {CODE_DIR} pull -q --ff-only
else:
    !git clone -q -b v2.1.1 https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)

from huggingface_hub import hf_hub_download
PUB = "/content/pub"
os.makedirs(PUB, exist_ok=True)
for f in ("best.pt", "tokenizer.json"):
    hf_hub_download("bonopassale/miniLLM_v2.1-49M-42500it_2.79Btoks_0.9ep_20261001", f, local_dir=PUB)
for n in ("A", "B", "C", "D", "Cs1", "Cs2"):
    p = f"{CKPT}/sft_{n}/final.pt"
    print("✓" if os.path.exists(p) else "✗ MANQUANT", p)


In [ ]:
# ── 2) Comparaison appariée (référence = modèle publié) ──
specs = [f"PUB={PUB}/best.pt"] + [f"{n}={CKPT}/sft_{n}/final.pt" for n in ("A", "B", "C", "D", "Cs1", "Cs2")]
ARGS = " ".join("--ckpt " + s for s in specs)
os.makedirs(f"{BASE}/evals_connaissances", exist_ok=True)
!python compare_knowledge.py {ARGS} --out_dir {BASE}/evals_connaissances --data_dir {DATA}


**Lecture**
- `Δ vs PUB` : écart en points par rapport au modèle publié. `+/-` : nombre de questions réussies par ce modèle seulement / par PUB seulement. `p` : test exact de McNemar (p < 0,05 : écart probablement pas dû au hasard).
- **A proche de C** → le gain vient surtout du pré-entraînement prolongé. **A proche de PUB** → il vient de la recette de SFT.
- C, Cs1, Cs2 ne diffèrent que par la graine : leur dispersion donne le bruit.

**À m'envoyer :** le tableau final.